In [1]:

import math
import tqdm
import torch
import gpytorch
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import NearestNeighbors
from imblearn.over_sampling import SMOTE 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from deep_gp.deep_kernel_class import LargeFeatureExtractor, GPRegressionModel

%matplotlib inline
%load_ext autoreload
%autoreload 2



In [2]:

features = pd.read_csv('../features.csv')
targets = pd.read_csv('../targets.csv')
data = features.merge(targets[['study_id','patient_id','case_ISUP']], on='study_id')
X = data.drop(['study_id','case_ISUP'], axis=1) # predictors
y = data['case_ISUP']  # target variable

In [3]:
df = X.copy() 
df["case_ISUP"] = y

idx_class0 = df.index[df["case_ISUP"] == 0]

# Storage for selected class-0 samples 
selected_class0 = []

# For each class 1–5, find nearest class-0 sample

for cls in range(1, 6): 
    print(f"Processing class {cls}") 
    idx_cls = df.index[df["case_ISUP"] == cls] 
    
    # Fit KNN on class 0 samples 
    knn = NearestNeighbors(n_neighbors=1) 
    knn.fit(df.loc[idx_class0, X.columns]) 
    
    # For each sample in class cls, find nearest class-0 sample 
    distances, indices = knn.kneighbors(df.loc[idx_cls, X.columns]) 
    # Convert neighbor indices to original dataframe indices 
    nearest_idx0 = idx_class0[indices.flatten()] 
    # Add to list 
    selected_class0.extend(nearest_idx0)

# keep only unique samples in class 0
selected_class0 = list(set(selected_class0)) 
print(f"\nOriginal class 0 count: {len(idx_class0)}") 
print(f"New class 0 count: {len(selected_class0)}")

# new reduced dataset
df_new = pd.concat([ df[df["case_ISUP"] != 0], # keep all minority classes 
                    df.loc[selected_class0] # keep only selected class-0 samples ])
])

print("\nFinal class counts:")
print(df_new["case_ISUP"].value_counts().sort_index())



# SMOTE only for classes 3, 4, 5

X_smote = df_new.drop(columns=["case_ISUP"])
y_smote = df_new["case_ISUP"]

sampling_strategy = {
    3: 150,
    4: 150,
    5: 150
}

sm = SMOTE(
    sampling_strategy=sampling_strategy,
    k_neighbors=3,
    random_state=42
)

X_resampled, y_resampled = sm.fit_resample(X_smote, y_smote)

df_resampled = pd.concat([
    pd.DataFrame(X_resampled, columns=X_smote.columns),
    pd.Series(y_resampled, name="case_ISUP")
], axis=1)

print("\nFinal class counts AFTER SMOTE:")
print(df_resampled["case_ISUP"].value_counts().sort_index())


Processing class 1
Processing class 2
Processing class 3
Processing class 4
Processing class 5

Original class 0 count: 589
New class 0 count: 262

Final class counts:
case_ISUP
0    262
1    157
2    154
3     69
4     27
5     35
Name: count, dtype: int64

Final class counts AFTER SMOTE:
case_ISUP
0    262
1    157
2    154
3    150
4    150
5    150
Name: count, dtype: int64


In [4]:
#Compute Spearman correlations on the final dataset

df = df_resampled.copy()

corr = df.corr(method="spearman")["case_ISUP"].abs().sort_values(ascending=False)
corr = corr.drop("case_ISUP")

top10 = corr.head(10).index.tolist()
top20 = corr.head(20).index.tolist()

print("\nTop 10 correlated features (Spearman):", top10)
print("Top 20 correlated features (Spearman):", top20)



Top 10 correlated features (Spearman): ['original_firstorder_Minimum', 'original_shape_Sphericity', 'original_shape_SurfaceVolumeRatio', 'original_gldm_LargeDependenceHighGrayLevelEmphasis', 'original_gldm_DependenceEntropy', 'original_ngtdm_Strength', 'original_shape_LeastAxisLength', 'original_ngtdm_Coarseness', 'original_glrlm_LongRunHighGrayLevelEmphasis', 'original_gldm_SmallDependenceLowGrayLevelEmphasis']
Top 20 correlated features (Spearman): ['original_firstorder_Minimum', 'original_shape_Sphericity', 'original_shape_SurfaceVolumeRatio', 'original_gldm_LargeDependenceHighGrayLevelEmphasis', 'original_gldm_DependenceEntropy', 'original_ngtdm_Strength', 'original_shape_LeastAxisLength', 'original_ngtdm_Coarseness', 'original_glrlm_LongRunHighGrayLevelEmphasis', 'original_gldm_SmallDependenceLowGrayLevelEmphasis', 'original_glszm_SmallAreaLowGrayLevelEmphasis', 'original_glcm_Imc2', 'original_shape_MeshVolume', 'original_shape_VoxelVolume', 'original_shape_MinorAxisLength', 'ori

In [ ]:

def run_dkl_experiment(feature_list, df):

    print(f"Running DKL with {len(feature_list)} features")

    # Data split 
    X = df[feature_list]
    y = df["case_ISUP"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Scaling 
    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled  = scaler_X.transform(X_test)

    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1,1)).ravel()
    y_test_scaled  = scaler_y.transform(y_test.values.reshape(-1,1)).ravel()

    train_x = torch.tensor(X_train_scaled, dtype=torch.float32)
    train_y = torch.tensor(y_train_scaled, dtype=torch.float32)
    test_x  = torch.tensor(X_test_scaled, dtype=torch.float32)
    test_y  = torch.tensor(y_test_scaled, dtype=torch.float32)

    if torch.cuda.is_available():
        train_x, train_y = train_x.cuda(), train_y.cuda()
        test_x, test_y = test_x.cuda(), test_y.cuda()


    likelihood = gpytorch.likelihoods.GaussianLikelihood()

    model = GPRegressionModel(train_x, train_y, likelihood, data_dim=train_x.shape[1])

    if torch.cuda.is_available():
        model = model.cuda()
        likelihood = likelihood.cuda()

    optimizer = torch.optim.Adam([
        {'params': model.feature_extractor.parameters()},
        {'params': model.covar_module.parameters()},
        {'params': model.mean_module.parameters()},
        {'params': likelihood.parameters()},
    ], lr=0.01)

    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    # Training 
    model.train()
    likelihood.train()

    for i in range(300):
        optimizer.zero_grad()
        output = model(train_x)
        loss = -mll(output, train_y)
        loss.backward()
        optimizer.step()

    #  Evaluation 
    model.eval()
    likelihood.eval()

    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        preds = likelihood(model(test_x))

    # Mean in scaled space
    y_pred_scaled = preds.mean.detach().cpu().numpy()
    y_true_scaled = test_y.detach().cpu().numpy()

    # Inverse-transform to original ISUP scale
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1,1)).ravel()
    y_true = scaler_y.inverse_transform(y_true_scaled.reshape(-1,1)).ravel()

    # uncertainty (std) in original ISUP scale 
    f_var_scaled = preds.variance.detach().cpu().numpy()
    f_std_scaled = np.sqrt(f_var_scaled)

    # y_scaler: y_scaled = (y - mean) / scale  =>  std_y = std_scaled * scale
    scale_y = scaler_y.scale_[0]
    f_std = f_std_scaled * scale_y   # std in original ISUP units

    # Build DataFrame for analysis
    df_unc = pd.DataFrame({
        "true": y_true,
        "pred": y_pred,
        "std": f_std
    })

    df_unc["true_class"] = np.round(df_unc["true"]).astype(int)

    # Metrics
    mse = mean_squared_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)

    print(f"Test MSE: {mse:.4f}")
    print(f"Test R²:  {r2:.4f}")

    # Per-class mean uncertainty (useful summary)
    per_class_std = df_unc.groupby("true_class")["std"].mean()
    print("\nMean predictive std per true ISUP class:")
    print(per_class_std)

    return mse, r2, df_unc


In [6]:
results = {}
uncertainties = {}

all_features = df.drop(columns=["case_ISUP"]).columns.tolist()

results["all_features"], r2_all, df_unc_all = run_dkl_experiment(all_features, df)
results["top20"], r2_20, df_unc_20 = run_dkl_experiment(top20, df)
results["top10"], r2_10, df_unc_10 = run_dkl_experiment(top10, df)

uncertainties["all_features"] = df_unc_all
uncertainties["top20"] = df_unc_20
uncertainties["top10"] = df_unc_10

print("\n===== FINAL COMPARISON =====")
print(f"all_features: MSE={results['all_features']:.4f}, R²={r2_all:.4f}")
print(f"top20:        MSE={results['top20']:.4f}, R²={r2_20:.4f}")
print(f"top10:        MSE={results['top10']:.4f}, R²={r2_10:.4f}")


Running DKL with 109 features


/home/chrysoula/.cache/pypoetry/virtualenvs/deep-gp-BifO3guL-py3.11/lib/python3.11/site-packages/linear_operator/utils/sparse.py:51: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  if nonzero_indices.storage():
/home/chrysoula/.cache/pypoetry/virtualenvs/deep-gp-BifO3guL-py3.11/lib/python3.11/site-packages/linear_operator/utils/sparse.py:66: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  res = cls(index_tensor, value_tensor, interp_size)
/home/chrysoula/.cache/pypoetry/virtualenvs/deep-gp-BifO3guL-py3.11/lib/python3.11/site-packages/linear_operator/

Test MSE: 1.4633
Test R²:  0.5413

Mean predictive std per true ISUP class:
true_class
0    0.659730
1    0.654716
2    0.676648
3    0.651318
4    0.652756
5    0.651962
Name: std, dtype: float64
Running DKL with 20 features
Test MSE: 2.0212
Test R²:  0.3664

Mean predictive std per true ISUP class:
true_class
0    0.606172
1    0.606588
2    0.603641
3    0.601380
4    0.600376
5    0.603037
Name: std, dtype: float64
Running DKL with 10 features
Test MSE: 2.1641
Test R²:  0.3216

Mean predictive std per true ISUP class:
true_class
0    0.834203
1    0.835107
2    0.837879
3    0.835179
4    0.831570
5    0.831018
Name: std, dtype: float64

===== FINAL COMPARISON =====
all_features: MSE=1.4633, R²=0.5413
top20:        MSE=2.0212, R²=0.3664
top10:        MSE=2.1641, R²=0.3216
